# 01b — Deep dives: resolving open semantic questions  (Phase 1, pre-flattening)

**This is a DECISION-manufacturing notebook, not open exploration.** Every section
ends with a **DECISION** cell stating the rule the Phase 2 pipeline will follow
(or **ESCALATE** / **OPEN — needs human input**). All decisions are also recorded
in `DECISIONS.md` at the repo root.

Rules of engagement: descriptive numbers only (tiny dev data — no thresholds, no
models); read-only on existing collections; reusable logic lives in `src/frauddet/`
(unit-tested), the notebook stays thin.

Sections:
1. Diagnose unmatched IDs
2. Casino UUID joinability
3. Status semantics
4. Fingerprint profile
5. Bonus schema
6. Ledger hunt
7. Uniqueness & duplicates
8. Currency rule

> The DB is live dev — exact counts drift between runs; every conclusion here is
> structural, not count-dependent.

## 0. Setup

In [1]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / "src"))

import pandas as pd

from frauddet import config, identity, io_mongo, ip_utils

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 40)

with io_mongo.get_database() as db:
    players     = pd.DataFrame(list(db["players"].find()))
    bets        = pd.DataFrame(list(db["bet_transactions"].find()))
    deposits    = pd.DataFrame(list(db["deposittransactions"].find()))
    withdrawals = pd.DataFrame(list(db["withdrawaltransactions"].find()))
    gateway     = pd.DataFrame(list(db["gateway-payment-transactions"].find()))
    bonuses     = pd.DataFrame(list(db["bonustransactions"].find()))
    activity    = pd.DataFrame(list(db["useractivitylogs"].find()))
    logins      = pd.DataFrame(list(db["loginlogs"].find()))

print({n: len(d) for n, d in [("players", players), ("bets", bets), ("deposits", deposits),
                              ("withdrawals", withdrawals), ("gateway", gateway),
                              ("bonuses", bonuses), ("activity", activity), ("logins", logins)]})

{'players': 78, 'bets': 379, 'deposits': 153, 'withdrawals': 77, 'gateway': 402, 'bonuses': 61, 'activity': 3841, 'logins': 855}


## 1. Diagnose unmatched IDs

Phase 1 found IDs in the transaction feeds that join to no `players` row. Before
the flatten layer can decide what to do with them, classify each one:
**(a) staff**, **(b) deleted/archived player**, **(c) obvious test pattern**
(`identity.looks_like_test_number`, unit-tested), **(d) unknown**.

First, can (a) or (b) even occur?

In [2]:
player_phones = set(players["username"].map(identity.normalize_phone).dropna())

# (a) staff: what do staff log in with?
staff = logins[logins["userType"] != "PLAYER"]
staff_phones = set(staff["loginId"].map(identity.normalize_phone).dropna())
print("staff loginIds:", sorted(set(staff["loginId"])))
print("staff loginIds that are phone-like:", sorted(staff_phones) or "NONE")

# (b) deleted players: do they vanish from the collection, or stay visible?
print("\nplayers.archived: ", players["archived"].value_counts(dropna=False).to_dict())
print("players.isDeleted:", players["isDeleted"].value_counts(dropna=False).to_dict())
deleted_phones = set(
    players.loc[(players["isDeleted"] == True) | (players["archived"] == True), "username"]
    .map(identity.normalize_phone).dropna()
)
print(f"\n-> soft-deleted players STAY VISIBLE in the collection ({len(deleted_phones)} of them),")
print("   so they joined like everyone else — an unmatched ID cannot be a deleted player.")

staff loginIds: ['adminmanager', 'amayoperator', 'amaysmanager', 'davidmanager', 'dummycompany', 'dummyoperatoradmin', 'jitenmanager', 'jitenops', 'jitensupes', 'operatordemo', 'satyamanager', 'satyaoperator', 'satyaops']
staff loginIds that are phone-like: NONE

players.archived:  {False: 78}
players.isDeleted: {False: 41, nan: 33, True: 4}

-> soft-deleted players STAY VISIBLE in the collection (4 of them),
   so they joined like everyone else — an unmatched ID cannot be a deleted player.


In [3]:
# Collect every normalized ID that fails to join, per feed.
feeds = {
    "bet_transactions": ("loginId", bets),
    "deposittransactions": ("userId", deposits),
    "gateway-payment-transactions": ("userId", gateway),
    "bonustransactions": ("userId", bonuses),
    "useractivitylogs": ("playerId", activity),
}
unmatched = {}
for name, (field, df) in feeds.items():
    norm = df[field].map(identity.normalize_phone)
    unmatched[name] = set(norm[~norm.isin(player_phones)].dropna())
    print(f"{name:30} {len(unmatched[name]):3} distinct unmatched IDs")

union = set().union(*unmatched.values())
print(f"\nunion across feeds: {len(union)} distinct IDs")

bet_transactions                 2 distinct unmatched IDs
deposittransactions              2 distinct unmatched IDs
gateway-payment-transactions    32 distinct unmatched IDs
bonustransactions                2 distinct unmatched IDs
useractivitylogs                 9 distinct unmatched IDs

union across feeds: 35 distinct IDs


In [4]:
# Classify each unmatched ID (priority: staff > deleted_player > test_pattern > unknown).
def classify(p):
    if p in staff_phones:
        return "staff"
    if p in deleted_phones:
        return "deleted_player"
    if identity.looks_like_test_number(p):
        return "test_pattern"
    return "unknown"

cls = {p: classify(p) for p in union}
CLASSES = ["staff", "deleted_player", "test_pattern", "unknown"]
table = pd.DataFrame(
    [{"collection": name, **{c: sum(1 for p in ids if cls[p] == c) for c in CLASSES},
      "total": len(ids)} for name, ids in unmatched.items()]
).set_index("collection")
table

,staff,deleted_player,test_pattern,unknown,total
collection,,,,,
bet_transactions,0,0,1,1,2
deposittransactions,0,0,1,1,2
gateway-payment-transactions,0,0,19,13,32
bonustransactions,0,0,1,1,2
useractivitylogs,0,0,6,3,9


In [5]:
# List the flagged IDs explicitly (nothing is excluded silently), plus extra evidence.
for c in ["test_pattern", "unknown"]:
    ids = sorted(p for p in union if cls[p] == c)
    print(f"{c} ({len(ids)}): {ids}\n")

# Evidence A: which unmatched IDs started (but never finished) registration?
with io_mongo.get_database() as db:
    otps = pd.DataFrame(list(db["playerresgistrationotps"].find()))
otp_phones = set(otps["contactNo"].map(identity.normalize_phone).dropna())
pre_reg = sorted(p for p in union if p in otp_phones)
print(f"present in playerresgistrationotps (pre-registration, never completed): {pre_reg}\n")

# Evidence B (caveat): how many REGISTERED players also match the test pattern?
flagged_real = sum(identity.looks_like_test_number(p) for p in player_phones)
print(f"CAVEAT: {flagged_real}/{len(player_phones)} registered players ALSO match the test "
      "pattern\n-> the heuristic is classification evidence, NEVER an exclusion key.\n")

# Evidence C: the IDs unmatched in CORE feeds (bets/deposits/bonuses) hold wallet accounts.
core = sorted(unmatched["bet_transactions"] | unmatched["deposittransactions"]
              | unmatched["bonustransactions"])
with io_mongo.get_database() as db:
    w = pd.DataFrame(list(db["walletaccounts"].find(
        {"userId": {"$in": [f"0{p}" for p in core] + core}})))
print(f"IDs unmatched in core money/bet feeds: {core}")
if len(w):
    print("...which nevertheless HAVE wallet accounts:")
    print(w[["userId", "amount", "status"]].to_string(index=False))

test_pattern (22): ['075333333', '6666699999', '751010101', '751111111', '751231231', '753222222', '753636363', '754222222', '754322222', '754333333', '755454545', '755555555', '756969696', '757417411', '757417412', '757575758', '757777777', '758528528', '758585858', '758787878', '788888888', '789789789']

unknown (13): ['751234569', '751530647', '752083028', '752301648', '754321012', '756248933', '758234901', '758234902', '758293146', '759535783', '759852364', '759876543', '759973863']



present in playerresgistrationotps (pre-registration, never completed): ['751010101', '751111111', '751231231', '754222222', '754321012', '756969696', '758528528']

CAVEAT: 60/78 registered players ALSO match the test pattern
-> the heuristic is classification evidence, NEVER an exclusion key.



IDs unmatched in core money/bet feeds: ['751231231', '754321012']
...which nevertheless HAVE wallet accounts:
    userId     amount status
0751231231 4993169.95 ACTIVE
0754321012   16720.00 ACTIVE


### DECISION — unmatched-ID policy (PROPOSED — needs human confirmation)

Classes (a) **staff** and (b) **deleted player** are empirically impossible here:
staff log in with usernames (none phone-like), and deletion is **soft** — deleted
players stay visible and join normally. Unmatched IDs are therefore
**pre-registration** (present in the OTP log), **test_pattern**, or **unknown**.

Flatten-layer policy (Phase 2):
1. **Keep every record.** Unjoined rows get `player_key = NULL` plus an
   `unjoined_class` tag ∈ {`pre_registration`, `test_pattern`, `unknown`}.
2. Unjoined rows are **excluded from the player feature table** (no player to
   profile) but fully counted in `unjoined_report.md` — never silently dropped.
3. `looks_like_test_number` stays advisory-only (most registered dev players match it too).

**OPEN — needs ops input:** the IDs unmatched in the core money/bet feeds hold
wallet accounts with material balances. Expected answer: internal test accounts.
If not, unregistered identities are transacting — a backend bug to flag.

## 2. Casino UUID joinability

`game_shown` / `game_clicked` / `game_played` carry a `player_id` that is not a
phone number. Can it be joined to anything we have?

In [6]:
SHOW = ["event_name", "player_id", "session_id", "game_id", "bet_transaction_id",
        "timestamp", "ip_address", "city", "country", "device_name", "origin"]
with io_mongo.get_database() as db:
    for coll in ["game_shown", "game_clicked", "game_played"]:
        print(f"--- {coll} ---")
        for doc in db[coll].find(limit=2):
            print({k: doc.get(k) for k in SHOW})
        print()

--- game_shown ---


{'event_name': 'game_shown', 'player_id': '0f27a2fe-665c-4d23-8f60-e26b72ac6f21', 'session_id': 'sess_web_01HXPW2WBCJQCG5GS3GQSMJ4SR', 'game_id': 'game_rocket_rivals', 'bet_transaction_id': None, 'timestamp': '2026-02-19T13:55:20.000Z', 'ip_address': None, 'city': None, 'country': None, 'device_name': 'MacBook Pro', 'origin': 'recommendation'}
{'event_name': 'game_shown', 'player_id': '0f27a2fe-665c-4d23-8f60-e26b72ac6f21', 'session_id': 'sess_web_01HXPW2WBCJQCG5GS3GQSMJ4SR', 'game_id': 'game_rocket_rivals', 'bet_transaction_id': None, 'timestamp': '2026-02-19T13:55:20.000Z', 'ip_address': None, 'city': None, 'country': None, 'device_name': 'MacBook Pro', 'origin': 'recommendation'}

--- game_clicked ---
{'event_name': 'game_clicked', 'player_id': '0f27a2fe-665c-4d23-8f60-e26b72ac6f21', 'session_id': 'sess_web_01HXPW2WBCJQCG5GS3GQSMJ4SR', 'game_id': 'game_rocket_rivals', 'bet_transaction_id': None, 'timestamp': '2026-02-19T13:55:21.123Z', 'ip_address': '203.0.113.42', 'city': 'Amsterda

{'event_name': 'game_played', 'player_id': '0f27a2fe-665c-4d23-8f60-e26b72ac6f21', 'session_id': 'sess_web_01HXPW2WBCJQCG5GS3GQSMJ4SR', 'game_id': 'game_rocket_rivals', 'bet_transaction_id': 'txn_01HXPW3ABCJQCG5GS3GQSMJ4XY', 'timestamp': '2026-02-19T13:56:05.000Z', 'ip_address': None, 'city': None, 'country': None, 'device_name': 'MacBook Pro', 'origin': 'recommendation'}
{'event_name': 'game_played', 'player_id': '0f27a2fe-665c-4d23-8f60-e26b72ac6f21', 'session_id': 'sess_web_01HXPW2WBCJQCG5GS3GQSMJ4SR', 'game_id': 'game_rocket_rivals', 'bet_transaction_id': 'txn_01HXPW3ABCJQCG5GS3GQSMJ4XY', 'timestamp': '2026-02-19T13:56:05.000Z', 'ip_address': None, 'city': None, 'country': None, 'device_name': 'MacBook Pro', 'origin': 'recommendation'}



In [7]:
with io_mongo.get_database() as db:
    pids, btx = set(), set()
    for coll in ["game_shown", "game_clicked", "game_played"]:
        pids |= {str(p) for p in db[coll].distinct("player_id") if p}
        btx |= {str(b) for b in db[coll].distinct("bet_transaction_id") if b}
    event_dates = sorted({str(t)[:10] for t in db["game_clicked"].distinct("timestamp")})

print(f"distinct game_* player_ids ({len(pids)}): {sorted(pids)}\n")
print("overlap with platform identifiers:")
print("  players._id (str):    ", len(pids & set(players["_id"].astype(str))))
print("  players.playerId:     ", len(pids & set(players["playerId"].astype(str))))
print("  players.username:     ", len(pids & set(players["username"].astype(str))))
print("  loginlogs.fingerprint:", len(pids & set(logins["fingerprint"].astype(str))))
print(f"\ngame_* bet_transaction_ids: {sorted(btx)}")
print("  vs bets.ticketId:      ", len(btx & set(bets["ticketId"].astype(str))))
print("  vs bets.txnTrackingId: ", len(btx & set(bets["txnTrackingId"].astype(str))))
print(f"\ngame_* event dates: {event_dates}")
print(f"platform data starts: {players['createdAt'].min()} (players.createdAt)")

distinct game_* player_ids (15): ['0f27a2fe-665c-4d23-8f60-e26b72ac6f21', '1244444444', '2333333333', '3216549870', '3355335533', '5097103780', '6546546546', '7555555555', '7888888888', '7891990356', '8777777777', '9518476230', '9638520741', '9777777777', '9873771606']

overlap with platform identifiers:
  players._id (str):     0
  players.playerId:      0
  players.username:      0
  loginlogs.fingerprint: 0

game_* bet_transaction_ids: ['txn_01HXPW3ABCJQCG5GS3GQSMJ4XY']
  vs bets.ticketId:       0
  vs bets.txnTrackingId:  0

game_* event dates: ['2026-02-19', '2026-03-13', '2026-03-16', '2026-03-18', '2026-03-24', '2026-03-25']
platform data starts: 2026-05-21 06:45:19.652000 (players.createdAt)


### DECISION — casino telemetry is NOT joinable → ESCALATE

Zero overlap between `game_*.player_id` and **any** platform identifier
(`players._id`, `playerId`, `username`, fingerprints). The IDs are foreign formats
(one UUID, the rest 10-digit strings unlike Ugandan numbers); event dates
**predate** the platform's data window; the one `bet_transaction_id` (`txn_…`
ULID-style) matches no bets key. These look like vendor sample/demo events.

**Exact escalation wording for the backend team:**
1. Casino games produce **no monetary bet records** anywhere in this DB. We need
   per-round records — player key, game id, round/ticket id, **stake, payout,
   currency**, timestamp — ideally written to `bet_transactions` with a casino
   `gameType`, or to a parallel collection sharing the same player key.
2. Casino telemetry (`game_*`) must be stamped with **`players._id`** (or at
   minimum `username`/`contactNo`). Today's `player_id` matches nothing.
3. Until both land, casino play is **invisible to fraud detection** — the
   "player-level profile across ALL games" goal is blocked on this.

## 3. Status semantics

What does each status actually mean, so Phase 2 can whitelist canonical sets?

In [8]:
# Bets: status x result, and where resultedDate actually appears.
print(pd.crosstab(bets["status"], bets["result"].fillna("<none>")))
print("\nrows with resultedDate, by status:")
print(bets.assign(has_rd=bets["resultedDate"].notna()).groupby("status")["has_rd"].sum().to_string())

settled = bets[bets["status"] == "SETTLED"]
gap_h = (settled["updatedAt"] - settled["createdDate"]).dt.total_seconds() / 3600
print("\nSETTLED bets: updatedAt minus createdDate, hours (settlement-time proxy):")
print(gap_h.describe().round(1).to_string())

result    <none>  LOSE  VOID  WIN
status                           
ACCEPTED      51     0     0    0
SETTLED        0   182     0  139
VOID           0     0     7    0

rows with resultedDate, by status:
status
ACCEPTED    0
SETTLED     0
VOID        7

SETTLED bets: updatedAt minus createdDate, hours (settlement-time proxy):
count    321.0
mean       5.4
std       13.8
min        0.1
25%        1.7
50%        3.0
75%        4.3
max      118.5


**Reading.** The mystery from Phase 1 ("result 96% populated but resultedDate 2%")
resolves cleanly: `result` is filled exactly for SETTLED+VOID bets (ACCEPTED bets
have none yet), while `resultedDate` is **only written for VOID bets**. For
settlement time the only candidate is `updatedAt` (median a few hours after
placement — plausible).

In [9]:
# Deposits: what each status looks like, and whether 'initiated' rows get a completed twin.
print(deposits["status"].value_counts().to_string())

cols = ["transactionId", "userId", "amount", "createdAt", "updatedAt"]
print("\nmanual_reconciliation samples:")
print(deposits[deposits["status"] == "manual_reconciliation"][cols].head(3).to_string(index=False))
fl = deposits[deposits["status"] == "failed"]
extra = [c for c in ["manualReconciliationReason", "reconciledBy"] if c in fl.columns]
print("\nfailed samples (note: reasons/reconcilers appear here):")
print(fl[cols + extra].head(2).to_string(index=False))

init = deposits[deposits["status"] == "initiated"]
comp = deposits[deposits["status"] == "completed"]
tw = init.merge(comp, on=["userId", "amount"], suffixes=("_i", "_c"))
n_twin = tw[tw["createdAt_c"] >= tw["createdAt_i"]]["transactionId_i"].nunique()
print(f"\ninitiated deposits: {len(init)} | with a later completed twin (same user+amount): {n_twin}")
print("-> deposit transactionIds are unique (see section 7): each attempt is its own doc;")
print("   'initiated' rows are mostly abandoned attempts, not pre-states of a later success.")

status
completed                131
initiated                 12
manual_reconciliation      6
failed                     4

manual_reconciliation samples:
             transactionId     userId  amount               createdAt               updatedAt
TXNY_MPTO775R_9F0AC4BA6978 0757575757   40000 2026-05-31 11:02:15.255 2026-05-31 14:00:00.073
TXNY_MPV024YC_E5DB2ADF90BE 0702133888  111111 2026-06-01 09:22:00.714 2026-06-01 12:10:00.045
TXNY_MPV02E8R_5D8B07951C1D 0702133888    1100 2026-06-01 09:22:12.609 2026-06-01 12:10:00.083

failed samples (note: reasons/reconcilers appear here):
             transactionId     userId  amount               createdAt               updatedAt                                                                                                                                                                                                                                 manualReconciliationReason reconciledBy
TXNY_MPTO9OTJ_EC833C2C3421 0757575757    6100 2026-05-

In [10]:
# Withdrawals: status x executionType (raw rows — lifecycle duplicates analysed in section 7).
pd.crosstab(withdrawals["status"], withdrawals["executionType"])

executionType,AUTO,MANUAL
status,,
completed,18,3
declined,0,4
failed,10,1
initiated,24,13
pending,0,4


### DECISION — canonical status whitelists (written to `config.yaml` in Phase 2)

| concept | rule |
|---|---|
| `money_in` | `deposittransactions.status == "completed"` |
| `money_out` | `withdrawaltransactions.status == "completed"` **after lifecycle dedup (§7)** |
| `pending_withdrawal` | deduped status ∈ {`initiated`, `pending`} |
| `settled_bet` | `bet_transactions.status == "SETTLED"` (result WIN/LOSE); VOID excluded from outcome stats; ACCEPTED = open bet |
| settlement time | `updatedAt` as proxy — `resultedDate` is only written for VOID. `# ASSUMED — verify with backend` |

**OPEN — needs ops input:** does `deposits.status == "manual_reconciliation"`
end up credited to the player? Excluded from `money_in` until confirmed.

## 4. Fingerprint profile

`loginlogs.fingerprint` exists at 100% coverage. Is it a real device hash, and is
sharing across players a usable multi-account signal? (PLAYER rows only.)

In [11]:
pl = logins[logins["userType"] == "PLAYER"]
fps = pl["fingerprint"].astype(str)

print("sample value:", fps.iloc[0])
print("\nlength distribution:", fps.str.len().value_counts().to_dict())
print("non-64-char values:", sorted(set(fps[fps.str.len() != 64])))
hexset = set("0123456789abcdef")
print("share of values that are pure lowercase hex:",
      round(fps.map(lambda s: set(s) <= hexset).mean(), 4))

sample value: 48b4e69c379c8c7cdd2b63fb2776e21414e8a741d2604556574e3125ab954a55

length distribution: {64: 448, 9: 2}
non-64-char values: ['not-found']
share of values that are pure lowercase hex: 0.9956


In [12]:
real = pl[fps.str.len() == 64]  # drop the 'not-found' sentinel rows
print(f"PLAYER login rows: {len(pl)} ({len(pl) - len(real)} with sentinel 'not-found')")
print(f"distinct fingerprints: {real['fingerprint'].nunique()} | distinct players logging in: {real['loginId'].nunique()}")

ppf = real.groupby("fingerprint")["loginId"].nunique()
print("\nplayers-per-fingerprint distribution (n_players -> n_fingerprints):")
print(ppf.value_counts().sort_index().to_dict())
print("\ntop 5 most-shared fingerprints (players on each):")
print(ppf.sort_values(ascending=False).head(5).to_string())

fpp = real.groupby("loginId")["fingerprint"].nunique()
print("\nfingerprints-per-player distribution (n_fps -> n_players):")
print(fpp.value_counts().sort_index().to_dict())

staff_fps = set(logins.loc[logins["userType"] != "PLAYER", "fingerprint"].astype(str))
print("\nfingerprints appearing in BOTH player and staff logins:",
      len(set(real["fingerprint"]) & staff_fps))

PLAYER login rows: 450 (2 with sentinel 'not-found')
distinct fingerprints: 29 | distinct players logging in: 70

players-per-fingerprint distribution (n_players -> n_fingerprints):
{1: 10, 2: 5, 4: 4, 5: 4, 6: 2, 8: 1, 11: 1, 18: 1, 28: 1}

top 5 most-shared fingerprints (players on each):
fingerprint
48b4e69c379c8c7cdd2b63fb2776e21414e8a741d2604556574e3125ab954a55    28
41d09edc1646efea9e18349b73755b6c5dc613533dc76b0700792197c4aa82f5    18
2fd75a67a9274a64a3ebd767decf951b72756193705ad366029dc7015401ea47    11
3ce037b7c11010bfe956551486c7d13a97683c6d1ed3d2c7f20e124351131c3a     8
3389cc58fe16df97160a32235a05534fecda1b3951ec7c9cd3d7c7053ae7163e     6

fingerprints-per-player distribution (n_fps -> n_players):
{1: 42, 2: 16, 3: 4, 4: 2, 5: 2, 6: 2, 8: 1, 9: 1}

fingerprints appearing in BOTH player and staff logins: 0


### DECISION — fingerprint is a STRONG multi-account feature (calibration is [PROD])

It is a real 64-char hex hash (SHA-256-like), stable per player (~60% of players
show exactly one), with a `"not-found"` sentinel to treat as missing, and zero
player↔staff crossover. Phase 3 builds `n_players_sharing_my_fingerprint` from
`loginlogs` filtered to `userType == "PLAYER"` and `len(fingerprint) == 64`.

Dev-data sharing counts (one hash covering ~28 players) are **office-machine
artifacts** of test-account creation — structurally the feature is sound, but its
weights/thresholds are placeholders until production data.

## 5. Bonus schema

Which bonus-abuse features can Phase 3 actually build?

In [13]:
with io_mongo.get_database() as db:
    bonus_colls = sorted(c for c in db.list_collection_names() if "bonus" in c.lower())
    bonus_frames = {c: pd.DataFrame(list(db[c].find())) for c in bonus_colls}

print("bonus-related collections:", bonus_colls, "\n")
for c, df in bonus_frames.items():
    print(f"--- {c} (n={len(df)}) ---")
    if len(df):
        cov = (df.notna().mean() * 100).round(0).astype(int)
        print("  fields:", {k: f"{v}%" for k, v in cov.items()})
    print()

bonus-related collections: ['bonus', 'bonusaccounts', 'bonustransactions', 'bonuswallets'] 

--- bonus (n=5) ---
  fields: {'_id': '100%', 'bonusCreation': '100%', 'bonusAllocation': '100%', 'bonusUsage': '100%', 'status': '100%', 'bonusID': '100%', 'createdBy': '100%', 'updatedBy': '100%', 'createdAt': '100%', 'updatedAt': '100%', '__v': '100%'}

--- bonusaccounts (n=20) ---
  fields: {'_id': '100%', 'bonusCategoryId': '100%', 'userId': '100%', 'bonusTypeId': '100%', 'subBonusTypeId': '100%', '__v': '100%', 'allocated': '100%', 'bonusObjId': '100%', 'contribution': '100%', 'createdAt': '100%', 'expiryDate': '100%', 'isSettled': '100%', 'locked': '100%', 'unlocked': '100%', 'updatedAt': '100%'}

--- bonustransactions (n=61) ---
  fields: {'_id': '100%', 'bonusTypeId': '100%', 'refTransId': '100%', 'bonusCategoryId': '100%', 'subBonusTypeId': '48%', 'userId': '100%', '__v': '100%', 'amount': '100%', 'createdAt': '100%', 'fromAccountId': '100%', 'toAccountId': '100%', 'transactionType': 

In [14]:
# bonustransactions: lifecycle types and where refTransId points.
print("transactionType:", bonuses["transactionType"].value_counts().to_dict())
print("bonusTypeId:     ", bonuses["bonusTypeId"].value_counts().to_dict())

link = pd.DataFrame({
    "type": bonuses["transactionType"],
    "ref_in_deposit_txnIds": bonuses["refTransId"].isin(set(deposits["transactionId"])),
    "ref_in_bet_ticketIds": bonuses["refTransId"].isin(set(bets["ticketId"])),
})
link.groupby("type").agg(n=("type", "size"),
                         pct_ref_is_deposit=("ref_in_deposit_txnIds", "mean"),
                         pct_ref_is_bet=("ref_in_bet_ticketIds", "mean")).round(2)

transactionType: {'RELEASE': 32, 'ALLOCATED': 20, 'CONSUME': 7, 'EXPIRED': 2}
bonusTypeId:      {'First Deposit': 61}


,n,pct_ref_is_deposit,pct_ref_is_bet
type,,,
ALLOCATED,20,1.0,0.0
CONSUME,7,0.0,1.0
EXPIRED,2,0.0,0.0
RELEASE,32,0.0,1.0


In [15]:
# bonus = the bonus DEFINITIONS, incl. wagering-requirement-style config.
bdef = bonus_frames["bonus"].iloc[0]
for blk in ["bonusCreation", "bonusAllocation", "bonusUsage"]:
    val = bdef[blk]
    print(f"{blk} keys: {sorted(val.keys()) if isinstance(val, dict) else val}\n")

bonusCreation keys: ['_id', 'affiliateUserName', 'allowTags', 'applicableTo', 'applyExpiry', 'bonusCode', 'bonusType', 'denyTags', 'endDate', 'optIn', 'startDate', 'tags']

bonusAllocation keys: ['ReferralBonusAmount', '_id', 'allocation', 'bonusAmount', 'cashbackMinOdds', 'cashbackMinStake', 'cashbackNoOfLegs', 'cashbackPercentage', 'depositReleasePercentage', 'maxBonusAmount', 'maxCashbackBonusAmount', 'minDepositAmount', 'releaseCriteria', 'releaseCriteriaPercentage', 'releaseCriteriaa']

bonusUsage keys: ['BonusPayout', 'MaxWinningAmount', 'MaxWinningPercentage', 'MinCumulativeOdds', 'MinSelectionOdds', 'MinStake', 'MinStakeAmount', 'NumberOfLegs', '_id']



### DECISION — bonus features buildable in Phase 3

- From `bonustransactions` (join `userId` phone; lineage `refTransId` →
  `deposits.transactionId` for ALLOCATED rows): `n_bonus_allocations`,
  `bonus_allocated_sum`, released / consumed / expired counts & amounts,
  allocation→release timing.
- From `bonusaccounts`: locked/unlocked state, `isSettled`, expiry waste.
- From `bet_transactions`: bonus-funded stake share (`stakeBonus` vs `stakeReal`).
- From `deposittransactions`: share of deposits carrying `bonusTagName`.
- Wagering config exists in `bonus.bonusUsage` / `bonusAllocation`
  (`minDepositAmount`, `MaxWinningPercentage`, `MinStake`, …) joinable via
  `bonusObjId` — usable for "max-extraction" style features later.
- Dev data has a single bonus type ("First Deposit") — **do not hardcode it**.

## 6. Ledger hunt

`bet_transactions` / `deposittransactions` / `withdrawaltransactions` reference
`fromAccountId` / `toAccountId` ObjectIds. What do they point to, and is there a
balance-history ledger?

In [16]:
# The DB's three views (so we never mistake a view for a base collection).
with io_mongo.get_database() as db:
    views = list(db["system.views"].find())
pd.DataFrame([{"view": v["_id"], "is a view on": v["viewOn"]} for v in views])

,view,is a view on
0,booktestdbgp_test.EventFeedTimeSeries,system.buckets.EventFeedTimeSeries
1,booktestdbgp_test.player_account_sta...,walletaccounts
2,booktestdbgp_test.bet_transaction_re...,bet_transactions


In [17]:
# Resolve EVERY from/to ObjectId against the candidate account collections.
account_colls = ["walletaccounts", "userbettingpools", "cashaccounts", "bonusaccounts",
                 "payoutaccounts", "lostaccounts", "betting-tax-payable-accounts",
                 "wht-payable-accounts", "bonuswallets"]
with io_mongo.get_database() as db:
    id_sets = {c: {str(i) for i in db[c].distinct("_id")} for c in account_colls}
    betlosts = pd.DataFrame(list(db["betlosts"].find()))
    payouts = pd.DataFrame(list(db["payouttransactions"].find()))

def where_is(oid):
    s = str(oid)
    return next((c for c, ids in id_sets.items() if s in ids), "UNRESOLVED")

for name, df in [("bets", bets), ("deposits", deposits), ("withdrawals", withdrawals)]:
    for col in ["fromAccountId", "toAccountId"]:
        vals = [v for v in df[col].dropna() if not isinstance(v, list)]
        targets = pd.Series([where_is(v) for v in vals]).value_counts().to_dict()
        print(f"{name}.{col:14} -> {targets}")

refs = set()
for df in (betlosts, payouts):
    for col in ["fromAccountId", "toAccountId"]:
        for row in df[col].dropna():
            if isinstance(row, list):
                refs |= {d.get("ref") for d in row if isinstance(d, dict)}
print("\nbetlosts/payouts embed account refs by NAME:", sorted(refs))

bets.fromAccountId  -> {'walletaccounts': 372}
bets.toAccountId    -> {'userbettingpools': 372}
deposits.fromAccountId  -> {'cashaccounts': 153}
deposits.toAccountId    -> {'walletaccounts': 153}
withdrawals.fromAccountId  -> {'walletaccounts': 49}
withdrawals.toAccountId    -> {'cashaccounts': 32, 'walletaccounts': 15}

betlosts/payouts embed account refs by NAME: ['BettingTaxPayableAccount', 'LostAccount', 'PayoutAccount', 'UserBettingPool', 'WHTPayableAccount', 'WalletAccount']


In [18]:
# Do the account collections hold history, or just current balances?
with io_mongo.get_database() as db:
    for c in ["walletaccounts", "cashaccounts", "payoutaccounts", "lostaccounts"]:
        df = pd.DataFrame(list(db[c].find()))
        print(f"{c:16} n={len(df):3}  cols={list(df.columns)}")
    wallets = pd.DataFrame(list(db["walletaccounts"].find()))

# walletaccounts.userId is POLYMORPHIC: players._id hex OR phone.
u = wallets["userId"].astype(str)
is_hex = u.map(identity.is_objectid_hex)
hex_match = u[is_hex].isin(set(players["_id"].astype(str)))
phone_match = wallets["userId"].map(identity.normalize_phone).isin(player_phones)
print(f"\nwalletaccounts: {len(wallets)} rows | hex-keyed: {int(is_hex.sum())} "
      f"(matching players._id: {int(hex_match.sum())}) | phone-keyed joins: {int((~is_hex & phone_match).sum())}")
orphans = wallets[~(u.isin(set(players["_id"].astype(str))) | phone_match)]
print("true orphans (neither form joins):")
print(orphans[["userId", "amount", "status"]].to_string(index=False))

walletaccounts   n= 33  cols=['_id', 'userId', 'amount', 'updatedAt', 'status', 'username', '__v', 'createdAt']
cashaccounts     n= 32  cols=['_id', 'userId', 'accountNumber', '__v', 'amount', 'createdAt', 'decrease', 'increase', 'paymentMethod', 'updatedAt']


payoutaccounts   n= 15  cols=['_id', 'userId', '__v', 'amount', 'createdAt', 'updatedAt']
lostaccounts     n= 20  cols=['_id', 'userId', '__v', 'amount', 'createdAt', 'updatedAt']



walletaccounts: 33 rows | hex-keyed: 29 (matching players._id: 29) | phone-keyed joins: 2
true orphans (neither form joins):
    userId     amount status
0751231231 4993169.95 ACTIVE
0754321012   16720.00 ACTIVE


### DECISION — ledger reality

- The account collections hold **current balances only** (one row per player per
  account type, no history rows) — the **transaction collections themselves are
  the event trail**, and they are already in scope.
- Money graph confirmed: deposit `cashaccounts → walletaccounts`; bet
  `walletaccounts → userbettingpools`; payout `pool → wallet`; lost stake
  `pool → lostaccounts`; withdrawal `wallet → cash` (taxes to the
  `*-payable-accounts`).
- `player_account_statement_report_view` and `bet_transaction_report_view` are
  **Mongo views** — the pipeline reads **base collections only**.
- `walletaccounts.userId` is **polymorphic** (mostly `players._id` hex, a few
  phones) → the Phase 2 identity mapper resolves **both** forms
  (`identity.is_objectid_hex` + `normalize_phone`). The two true orphans are the
  same unknowns flagged in §1.

## 7. Uniqueness & duplicates

Can Phase 2 trust `transactionId` / `ticketId` as primary keys?

In [19]:
rows = []
for name, df, key in [("deposittransactions", deposits, "transactionId"),
                      ("withdrawaltransactions", withdrawals, "transactionId"),
                      ("gateway-payment-transactions", gateway, "transactionId"),
                      ("bet_transactions", bets, "ticketId")]:
    rows.append({"collection": name, "key": key, "rows": len(df),
                 "distinct_keys": df[key].nunique(),
                 "duplicated_rows": int(df[key].duplicated().sum())})
pd.DataFrame(rows).set_index("collection")

,key,rows,distinct_keys,duplicated_rows
collection,,,,
deposittransactions,transactionId,153,153,0
withdrawaltransactions,transactionId,77,42,35
gateway-payment-transactions,transactionId,402,402,0
bet_transactions,ticketId,379,379,0


In [20]:
# Withdrawals: duplicates are LIFECYCLE documents sharing one transactionId.
dup = withdrawals[withdrawals["transactionId"].duplicated(keep=False)]
status_sets = dup.groupby("transactionId")["status"].agg(lambda s: " + ".join(sorted(set(s))))
print("status combinations among duplicated withdrawal transactionIds:")
print(status_sets.value_counts().to_string())

swept = dup[dup["status"] == "failed"]
print("\nupdatedAt clock times of the 'failed' rows (batch-sweep evidence):")
print(swept["updatedAt"].dt.strftime("%H:%M").value_counts().head(5).to_string())

status combinations among duplicated withdrawal transactionIds:
status
completed + initiated              13
failed + initiated                 10
declined + initiated                4
completed + initiated + pending     3
failed + initiated + pending        1

updatedAt clock times of the 'failed' rows (batch-sweep evidence):
updatedAt
13:00    6
14:00    2
12:30    1
12:50    1
13:50    1


In [21]:
# Exact-duplicate rows and (user, timestamp) collisions.
for name, df in [("deposits", deposits), ("withdrawals", withdrawals),
                 ("bets", bets), ("gateway", gateway)]:
    exact = int(df.drop(columns=["_id"]).astype(str).duplicated().sum())
    print(f"{name}: exact duplicate rows (excluding _id): {exact}")
for name, df in [("deposits", deposits), ("withdrawals", withdrawals)]:
    n = int((df.groupby(["userId", "createdAt"]).size() > 1).sum())
    print(f"{name}: (userId, createdAt) collisions: {n}")

deposits: exact duplicate rows (excluding _id): 0
withdrawals: exact duplicate rows (excluding _id): 0
bets: exact duplicate rows (excluding _id): 0
gateway: exact duplicate rows (excluding _id): 0
deposits: (userId, createdAt) collisions: 0
withdrawals: (userId, createdAt) collisions: 0


### DECISION — dedup rules for the flatten layer

- **`withdrawaltransactions`: dedup required.** Lifecycle docs share a
  `transactionId` (initiated+completed, or initiated+failed — stale ones swept to
  `failed` by an on-the-hour batch job). Rule: per `transactionId` keep the
  **most-advanced status** (`completed > declined > failed > pending > initiated`),
  tie-break latest `updatedAt`. Status counting happens **after** dedup.
- Deposits, gateway, bets: keys unique — no dedup.
- No exact-duplicate rows and no `(user, timestamp)` collisions anywhere.

## 8. Currency rule

Bets looked INR while money looked UGX. Confirm per collection, including the
amount-bearing collections with no currency field at all.

In [22]:
check = ["bet_transactions", "deposittransactions", "withdrawaltransactions",
         "gateway-payment-transactions", "bonustransactions", "payouttransactions",
         "betlosts", "walletaccounts", "cashaccounts", "bonuswallets",
         "userbettingpools", "payoutaccounts", "lostaccounts"]
rows = []
with io_mongo.get_database() as db:
    for c in check:
        df = pd.DataFrame(list(db[c].find()))
        if "currency" in df.columns:
            note = str(df["currency"].value_counts(dropna=False).to_dict())
        elif "amount" in df.columns:
            note = "has amount, NO currency field"
        else:
            note = "no amount field"
        rows.append({"collection": c, "currency": note})
pd.DataFrame(rows).set_index("collection")

,currency
collection,
bet_transactions,{'INR': 379}
deposittransactions,{'UGX': 153}
withdrawaltransactions,{'UGX': 77}
gateway-payment-transactions,"{'UGX': 398, nan: 4}"
bonustransactions,"has amount, NO currency field"
payouttransactions,"has amount, NO currency field"
betlosts,"has amount, NO currency field"
walletaccounts,"has amount, NO currency field"
cashaccounts,"has amount, NO currency field"


### DECISION — currency constraint (binding for Phase 3 feature design)

Confirmed: **bets are INR; all money movement is UGX**; ~10 amount-bearing
collections carry **no currency field** (platform default assumed —
`# ASSUMED — verify`). Rule set:

1. Amounts are comparable **within one currency only**. Money↔money ratios
   (withdrawal_sum / deposit_sum, both UGX) are fine.
2. **Bets↔money ratios are FORBIDDEN** until production confirms a single
   currency or provides FX — this blocks PLAN.md Phase 3 features
   `turnover ratio (total_staked / total_deposited)` and any bonus-stake vs
   deposit comparison. Design the columns, fill with NaN + a blocked marker.
3. Every flattened table keeps its `currency` column; no silent cross-currency sums.

## FINDINGS — Notebook 01b: deep dives — 2026-06-12

- Verified: staff log in with usernames, never phones; player deletion is soft
  (isDeleted=True rows stay visible and join); unmatched IDs split into
  pre-registration (in OTP log), test-pattern, unknown — with the unknowns in core
  feeds holding wallet accounts with material balances. Casino `game_*` player_ids
  overlap NOTHING (0 matches vs players._id / playerId / username / fingerprints);
  event dates predate platform data. Bets: result is filled exactly for
  SETTLED+VOID; resultedDate only for VOID; settlement-time proxy = updatedAt.
  Withdrawal duplicates are lifecycle docs (initiated+completed / initiated+failed,
  hourly sweep); deposits/gateway/bets keys are unique. Fingerprint = real 64-hex
  hash, stable per player, "not-found" sentinel, zero player↔staff crossover.
  Bonus stack = definitions (`bonus`) + ledger (`bonustransactions`:
  ALLOCATED/RELEASE/CONSUME/EXPIRED, refTransId → deposit txnId) + state
  (`bonusaccounts`) + balance (`bonuswallets`). Account collections = current
  balances only; money graph mapped; the two report_* collections are Mongo VIEWS.
  Currency: bets INR, money UGX, many collections currency-less.
- Surprises: `walletaccounts.userId` is POLYMORPHIC — mostly `players._id` hex
  strings, a few phones (only collection like this); `resultedDate` written only
  for VOID bets; withdrawal lifecycle duplicates; casino events look like vendor
  demo data (foreign IDs, pre-launch dates).
- Gaps: casino has no monetary records and no player link (ESCALATE — exact
  wording in §2); no balance-history ledger (current balances only — event trail
  = transaction collections, acceptable); `manual_reconciliation` deposit
  semantics unknown; ~10 amount collections have no currency field.
- Decisions needed: (1) confirm unmatched-ID policy + identify the two unknowns
  holding wallets (§1); (2) send the casino logging escalation (§2); (3) ops:
  does manual_reconciliation credit the player? (§3); (4) backend: confirm
  updatedAt-as-settlement-time and the missing-currency default (§3, §8).
- Updated assumptions: CLAUDE.md gains — polymorphic wallet keying; status
  whitelists; withdrawal dedup; currency constraint (blocks turnover-style
  cross-currency features); views never read; fingerprint usable-strong.
  All eight decisions recorded in DECISIONS.md.

*(Counts in this notebook reflect the live dev DB at run time and drift between
runs; every decision above is structural, not count-dependent.)*